# SABRE: Calculation of the effective Hamiltonian elements
Before studying this notebook, it is recommended to go through the following:
- sabre_3spin_analytical_cosine.ipynb
- sabre_3spin_analytical_square.ipynb

This notebook deals with a three-spin model of a SABRE complex comprising hydride protons and a $^{19}\mathrm{F}$ nucleus from an equatorial 3-fluoropyridine ligand. Two periodic polarisation transfer fields are considered: sinusoidal and square-modulated. These are characterised by an static offset field $B_0$ and a time-dependent part with an amplitude of $B_1$ and time period $T$. An effective, time-independent Hamiltonian is constructed exactly numerically. Off-diagonal elements of the exact Hamiltonian are compared to the approximate values obtained from the analytical formulas that were derived in the previous notebooks.

In [ ]:
import spinguin as sg
sg.parameters.sparse_operator = False
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm, logm
from scipy.special import jv

### Create helper functions for the construction of the numerical time-independent Hamiltonian
- **hamiltonian**: Creates a static-field Hamiltonian at a given magnetic field $B_0$.
- **hamiltonian_eff_cos**: Creates the effective Hamiltonian when using a cosine-modulated magnetic field.
- **hamiltonian_eff_square**: Creates the effective Hamiltonian when using a square-modulated magnetic field.
- **transform_basis**: Transforms the Hamiltonian to a basis set where the hydride protons are expressed using singlet-triplet and the $^{19}\mathrm{F}$ using Zeeman eigenstates.

In [ ]:
def hamiltonian(ss:sg.SpinSystem, B0:float) -> np.ndarray:

    H_Z = 0
    for i in range(ss.nspins):
        H_Z = H_Z - ss.gammas[i]*B0*sg.operator(ss, f"I(z,{i})")
    
    H_CS = 0
    for i in range(ss.nspins):
        H_CS = H_CS - 1e-6*ss.chemical_shifts[i]*ss.gammas[i]*B0*sg.operator(ss, f"I(z,{i})")

    H_J = 0
    for i in range(ss.nspins):
        for j in range(i):
            IxSx = sg.operator(ss, f"I(x,{j})*I(x,{i})")
            IySy = sg.operator(ss, f"I(y,{j})*I(y,{i})")
            IzSz = sg.operator(ss, f"I(z,{j})*I(z,{i})")
            IiIj = IxSx + IySy + IzSz
            H_J = H_J + 2*np.pi*ss.J_couplings[i, j]*IiIj

    H = H_Z + H_CS + H_J

    return H

def hamiltonian_eff_cos(ss:sg.SpinSystem, B0:float, B1:float, nterms:int, T:float) -> np.ndarray:

    dt = T / nterms
    Bt = B0 + B1 * np.array([np.cos(i*2*np.pi / nterms) for i in range(nterms)])

    P = np.eye(np.prod(ss.mults))
    for i in range(nterms):
        H = hamiltonian(ss, Bt[i])
        P = expm(-1j*H*dt) @ P

    H_eff = 1j*logm(P)/T

    return H_eff

def hamiltonian_eff_square(ss:sg.SpinSystem, B0:float, B1:float, T:float) -> np.ndarray:

    dt = T/2

    H1 = hamiltonian(ss, B0+B1)
    H2 = hamiltonian(ss, B0-B1)

    P1 = expm(-1j*H1*dt)
    P2 = expm(-1j*H2*dt)

    P = P1 @ P2

    H_eff = 1j*logm(P)/T

    return H_eff

def transform_basis(H:np.ndarray) -> np.ndarray:

    a = np.array([[1], [0]])
    b = np.array([[0], [1]])

    S = np.sqrt(1/2) * (np.kron(a, b) - np.kron(b, a))
    Tp = np.kron(a, a)
    T0 = np.sqrt(1/2) * (np.kron(a, b) + np.kron(b, a))
    Tm = np.kron(b, b)

    Sa = np.kron(S, a)
    Sb = np.kron(S, b)
    Tpa = np.kron(Tp, a)
    Tpb = np.kron(Tp, b)
    T0a = np.kron(T0, a)
    T0b = np.kron(T0, b)
    Tma = np.kron(Tm, a)
    Tmb = np.kron(Tm, b)
    
    U = np.hstack((Sa, Tpb, T0a, Sb, Tma, T0b, Tpa, Tmb))

    H = np.linalg.inv(U) @ H @ U

    return H

### Create the spin system
The spin system consisting of the hydride protons and the $\mathrm{^{19}F}$ in the target molecule is created. Chemical shifts and $J$-couplings are assigned.

In [ ]:
ss = sg.SpinSystem(["1H", "1H", "19F"])
ss.chemical_shifts = [-23.1, -23.1, -124.2]
ss.J_couplings = [
    [ 0,    0,    0],
    [-7.67, 0,    0],
    [-0.05, 2.08, 0]
]

### Calculate the numerical effective Hamiltonians
Effective Hamiltonians as a function of the amplitude $B_1$ are calculated using both cosine and square waves with the period chosen to be $T = 10\ \mathrm{ms}$. Two cases are considered:
- $m=0$: In this case, the offset field $B_0 = 3.26$ µT.
- $m=1$: In this case, the offset field $B_0 = 43.1$ µT.

The parameter `nterms` defines how many points are used to describe the cosine wave.

**NOTE:** This step may take up to 10 minutes.

In [ ]:
# Define properties for the oscillating field
B1 = np.arange(1, 400)*1e-6
T = 10e-3
nterms = 100

# Case m=0
B0 = 3.26e-6

H_eff_cos_m0 = np.zeros((np.prod(ss.mults), np.prod(ss.mults), len(B1)), dtype=complex)
H_eff_square_m0 = np.zeros((np.prod(ss.mults), np.prod(ss.mults), len(B1)), dtype=complex)

for i, B1i in enumerate(B1):

    H_eff_cos = hamiltonian_eff_cos(ss, B0, B1i, nterms, T)
    H_eff_cos = transform_basis(H_eff_cos)
    H_eff_cos_m0[:,:,i] = H_eff_cos

    H_eff_square = hamiltonian_eff_square(ss, B0, B1i, T)
    H_eff_square = transform_basis(H_eff_square)
    H_eff_square_m0[:,:,i] = H_eff_square

# Case m=1
B0 = 43.1e-6

H_eff_cos_m1 = np.zeros((np.prod(ss.mults), np.prod(ss.mults), len(B1)), dtype=complex)
H_eff_square_m1 = np.zeros((np.prod(ss.mults), np.prod(ss.mults), len(B1)), dtype=complex)

for i, B1i in enumerate(B1):

    H_eff_cos = hamiltonian_eff_cos(ss, B0, B1i, nterms, T)
    H_eff_cos = transform_basis(H_eff_cos)
    H_eff_cos_m1[:,:,i] = H_eff_cos

    H_eff_square = hamiltonian_eff_square(ss, B0, B1i, T)
    H_eff_square = transform_basis(H_eff_square)
    H_eff_square_m1[:,:,i] = H_eff_square

### Compute the off-diagonal elements using the analytical expression
We compute the off-diagonal elements as a function of $B_1$ for $m=0$ and $m=1$. For the sinusoidal field, we use the formula
$$
\bra{S\beta}\tilde{\tilde{H}}^{(1)}\ket{T_-\alpha} = \frac{1}{2}a_xJ_m\left( \frac{a_{z1}}{\omega_p} \right)
$$
and, for the square-modulated field, we use
$$
\bra{S\beta}\tilde{\tilde{H}}^{(1)}\ket{T_-\alpha} = \frac{i a_x a_{z1} \omega_p \left[ e^{i\pi \left( m + \frac{a_{z1}}{\omega_p} \right)} - 1 \right]}{2\pi \left( m^2\omega_p^2 - a_{z1}^2 \right)}
$$
Please refer to the previous notebooks for the derivation of these formulas and the explanation of the symbols.

In [ ]:
ax = -np.sqrt(2)*np.pi*(ss.J_couplings[2,1] - ss.J_couplings[2,0])
az1 = -B1*(
    ss.gammas[0]*(1 + ss.chemical_shifts[0]*1e-6) + \
    ss.gammas[1]*(1 + ss.chemical_shifts[1]*1e-6) - \
    2*ss.gammas[2]*(1 + ss.chemical_shifts[2]*1e-6)
    ) / 2
wp = 2*np.pi / T

cos_analytical_m0 = np.array([(1/2) * ax * jv(0, az1i/wp) for az1i in az1])
cos_analytical_m1 = np.array([(1/2) * ax * jv(1, az1i/wp) for az1i in az1])

squ_analytical_m0 = np.array([
    (1j*ax*az1i*wp*(np.exp(1j*np.pi*(0+az1i/wp))-1)) / (2*np.pi*(0**2 * wp**2-az1i**2)) for az1i in az1
])
squ_analytical_m1 = np.array([
    (1j*ax*az1i*wp*(np.exp(1j*np.pi*(1+az1i/wp))-1)) / (2*np.pi*(1**2 * wp**2-az1i**2)) for az1i in az1
])

### Plot the off-diagonal elements and save the results
Plot the absolute values of the off-diagonal elements that drive the transition between $\ket{S\beta}$ and $\ket{T_-\alpha}$ as a function of $B_1$. The results are saved as .csv files.

In [ ]:
plt.plot(B1*1e6, np.abs(H_eff_cos_m0[3,4]), label="Cosine Numerical (m=0)")
plt.plot(B1*1e6, np.abs(H_eff_square_m0[3,4]), label="Square Numerical (m=0)")
plt.plot(B1*1e6, np.abs(cos_analytical_m0), label="Cosine Analytical (m=0)")
plt.plot(B1*1e6, np.abs(squ_analytical_m0), label="Square Analytical (m=0)")
plt.plot(B1*1e6, np.abs(H_eff_cos_m1[3,4]), label="Cosine Numerical (m=1)")
plt.plot(B1*1e6, np.abs(H_eff_square_m1[3,4]), label="Square Numerical (m=1)")
plt.plot(B1*1e6, np.abs(cos_analytical_m1), label="Cosine Analytical (m=1)")
plt.plot(B1*1e6, np.abs(squ_analytical_m1), label="Square Analytical (m=1)")
plt.xlabel(r"$B_1$ (µT)")
plt.ylabel("Coupling strength (rad/s)")
plt.legend()
plt.show()
plt.clf()

np.savetxt("m0_cosine_numerical.csv", np.abs(H_eff_cos_m0[3,4]), delimiter=",")
np.savetxt("m0_square_numerical.csv", np.abs(H_eff_square_m0[3,4]), delimiter=",")
np.savetxt("m0_cosine_analytical.csv", np.abs(cos_analytical_m0), delimiter=",")
np.savetxt("m0_square_analytical.csv", np.abs(squ_analytical_m0), delimiter=",")
np.savetxt("m1_cosine_numerical.csv", np.abs(H_eff_cos_m1[3,4]), delimiter=",")
np.savetxt("m1_square_numerical.csv", np.abs(H_eff_square_m1[3,4]), delimiter=",")
np.savetxt("m1_cosine_analytical.csv", np.abs(cos_analytical_m1), delimiter=",")
np.savetxt("m1_square_analytical.csv", np.abs(squ_analytical_m1), delimiter=",")